# NB05 — Baego RGB × multispectral × DSM pilot (Nadine exhaustive circles)

**Goal:** controlled comparison of four ResNet18 U-Net input configurations: **RGB (3)**, **RGB+DSM (4)**, **RGB+MS (7)**, and **RGB+MS+DSM (8)**. Use **only** the seven exhaustive Baego circles recorded by NB01; no other annotated polygons are used for training or evaluation. Labels come from the exhaustive positive Musanga polygons **clipped to each training/validation circle**.

All four configurations use exactly the same **intersection of valid RGB, four MS bands, DSM and reference labels**. Existing NB01 circle-fold assignments are used for three spatial folds; whole circles held out, no patch leakage; same optimizer, schedule and seed per fold. MS alignment checked against Baego RGB. DSM is lazily reprojected to that grid. Unseen-fold normalization uses training pixels only.

**Important:** This is a small seven-circle exploratory ablation, **not** a definitive independent validation of multimodal gains. Missing MS/DSM coverage may exclude circles; the notebook stops rather than silently replacing them. Input RGB/MS/DSM acquisition dates and alignment must be verified. Run section 02 diagnostics before enabling training.


## 01 — Environment, audited references and file paths


In [ ]:
%pip -q install rasterio geopandas pyogrio shapely segmentation-models-pytorch scikit-learn matplotlib pandas
from pathlib import Path
import json, re, random, gc, hashlib, time
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.windows import Window, from_bounds, transform as win_transform
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.features import rasterize
from shapely.geometry import box
from scipy.ndimage import binary_dilation
from IPython.display import display
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import r2_score, mean_absolute_error
import matplotlib.pyplot as plt
from google.colab import drive

if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final')
assert ROOT.exists(),ROOT
OUT=ROOT/'Outputs/V3/05_Baego_multisensor_pilot'
OUT.mkdir(parents=True,exist_ok=True)
A=json.loads((ROOT/'Outputs/V3/00_input_audit/00_input_manifest.json').read_text())
REF=ROOT/'Outputs/V3/01_reference_data_no_MOS34'
LINKS=pd.read_csv(REF/'01_circle_spatial_folds.csv')
BAEGO=LINKS[LINKS.site=='Baego'].copy()
assert BAEGO.circle_id.nunique()==7 and set(BAEGO.fold.astype(int))=={0,1,2}
RGB=ROOT/'Data/Ortho_RGB/Baego.tif'
MS=ROOT/'Data/MS/Baego_MS_4band_RGB_aligned.tif'
assert RGB.exists() and MS.exists(),(RGB,MS)
# Auto-discover DSM; if multiple candidates exist, set DSM explicitly after inspecting output.
DSM=None
dsm_candidates=sorted(p for p in (ROOT/'Data').rglob('*.tif')
    if re.search(r'(?:dsm|surface.model)',p.name,re.I)
    and 'Baego' in str(p) and p!=RGB and p!=MS)
print('DSM candidates:',*[str(p) for p in dsm_candidates],sep='\n  ')
if len(dsm_candidates)==1: DSM=dsm_candidates[0]
if DSM is None:
    print('ACTION: assign DSM=Path("...Baego DSM...tif") in this cell, then rerun.')
print('Seven Baego circles by fold:',BAEGO.groupby('fold').size().to_dict())
print('RGB:',RGB,'\nMS:',MS,'\nDSM:',DSM)


## 02 — Input and common-coverage audit (run before any GPU training)


In [ ]:
assert DSM is not None and Path(DSM).exists(), 'Set DSM explicitly in Section 01 and rerun.'
CIRCLES=gpd.read_file(A['exhaustive_annotations']['complete_roi_boundaries'])
POS=gpd.read_file(A['exhaustive_annotations']['positive_polygons'])
assert len(CIRCLES)==34
with rasterio.open(RGB) as ds:
    GRID=dict(crs=ds.crs,transform=ds.transform,width=ds.width,height=ds.height)
    assert ds.count>=3
with rasterio.open(MS) as ds:
    assert ds.crs==GRID['crs'] and ds.transform==GRID['transform']
    assert (ds.width,ds.height)==(GRID['width'],GRID['height']) and ds.count==4
    assert tuple(str(x).lower() for x in ds.descriptions)==('red','green','rededge','nir')
with rasterio.open(DSM) as ds:
    assert ds.crs is not None and ds.count>=1
    print('DSM CRS/resolution/NoData:',ds.crs,ds.res,ds.nodata)
# Preserve original feature row numbering: NB01 circle_id refers to the
# original shapefile row, NOT to a reordered or filtered GeoDataFrame.
CIRCLES['circle_id']=np.arange(len(CIRCLES))
CIRCLES=CIRCLES[CIRCLES.circle_id.isin(BAEGO.circle_id.astype(int))].copy()
assert len(CIRCLES)==7
from shapely import make_valid
from shapely.ops import unary_union

def safe_polygon(g,kind,ident,max_area_change=0.01):
    if g is None or g.is_empty:
        raise ValueError(f'{kind} {ident}: missing geometry')
    if g.is_valid: return g
    repaired=make_valid(g)
    parts=[]
    def extract(x):
        if x.geom_type in ('Polygon','MultiPolygon'):
            parts.append(x)
        elif hasattr(x,'geoms'):
            for y in x.geoms: extract(y)
    extract(repaired)
    if not parts: raise ValueError(f'{kind} {ident}: no polygon after repair')
    fixed=unary_union(parts)
    if fixed.is_empty or not fixed.is_valid:
        raise ValueError(f'{kind} {ident}: repair unsuccessful')
    area_change=abs(fixed.area-g.area)/max(fixed.area,g.area,1e-12)
    print(f'REPAIRED {kind} {ident}: relative area change {area_change:.5%}',flush=True)
    if area_change>max_area_change:
        raise ValueError(f'{kind} {ident}: area change {area_change:.3%} > {max_area_change:.0%}; inspect in GIS')
    return fixed

CIRCLES.geometry=[safe_polygon(g,'circle',int(i)) for i,g in
                  zip(CIRCLES.circle_id,CIRCLES.geometry)]
# Only positives intersecting Baego circles are needed: avoid failing
# because of unrelated invalid polygons elsewhere in the Congo project.
baego_footprint=unary_union(CIRCLES.geometry)
POS=POS[POS.intersects(baego_footprint)].copy()
assert len(POS)>0,'No Musanga positive polygons intersect Baego circles'
POS.geometry=[safe_polygon(g,'positive',int(i)) for i,g in
              zip(POS.index,POS.geometry)]
CIRCLES=CIRCLES.to_crs(GRID['crs'])
POS=POS.to_crs(GRID['crs'])
assert CIRCLES.geometry.is_valid.all() and POS.geometry.is_valid.all()
print('Valid geometry: Baego circles',len(CIRCLES),'positive polygons',len(POS))
FOLD_BY_ID=dict(zip(BAEGO.circle_id.astype(int),BAEGO.fold.astype(int)))
POS_SINDEX=POS.sindex

def read_patch(win, rgb_ds, ms_ds, dsm_vrt):
    rgb=np.ma.array(rgb_ds.read([1,2,3],window=win,masked=True))
    ms=np.ma.array(ms_ds.read([1,2,3,4],window=win,masked=True))
    z=np.ma.array(dsm_vrt.read(1,window=win,masked=True))
    rgbv=~np.any(np.ma.getmaskarray(rgb),axis=0)
    rgbv &= np.any(rgb.filled(0)!=0,axis=0)&np.any(rgb.filled(0)!=255,axis=0)
    msv=~np.any(np.ma.getmaskarray(ms),axis=0)
    msv &= np.all(np.isfinite(ms.filled(np.nan)),axis=0)
    # Source MS files declare nodata=None. Zero is treated conservatively as invalid,
    # subject to checking source transparency / reflectance scaling.
    msv &= np.all(ms.filled(0)>0,axis=0)
    zv=(~np.ma.getmaskarray(z))&np.isfinite(z.filled(np.nan))
    v=rgbv&msv&zv
    x=np.concatenate([rgb.filled(0).astype('float32')/255.,
                      ms.filled(0).astype('float32'),
                      z.filled(0).astype('float32')[None]],axis=0)
    return x,v

def label_patch(win,circle_geom):
    h,w=int(win.height),int(win.width)
    tf=win_transform(win,GRID['transform'])
    cm=rasterize([(circle_geom,1)],(h,w),transform=tf,fill=0,dtype='uint8').astype(bool)
    bbox=box(*rasterio.windows.bounds(win,GRID['transform']))
    hits=POS_SINDEX.query(bbox,predicate='intersects')
    gt=((POS.geometry.iloc[i],1) for i in hits)
    yy=rasterize(gt,(h,w),transform=tf,fill=0,dtype='uint8').astype('float32')
    return yy,cm

def circle_windows(geom, crop=256, stride=192):
    raw=from_bounds(*geom.bounds,transform=GRID['transform'])
    c0=max(0,int(np.floor(raw.col_off)));r0=max(0,int(np.floor(raw.row_off)))
    c1=min(GRID['width'],int(np.ceil(raw.col_off+raw.width)))
    r1=min(GRID['height'],int(np.ceil(raw.row_off+raw.height)))
    if c1-c0<crop or r1-r0<crop: return []
    def locs(a,b):
        last=b-crop
        return sorted(set(list(range(a,last+1,stride))+[last]))
    return [Window(c,r,crop,crop) for r in locs(r0,r1) for c in locs(c0,c1)]

COVERAGE=[]
with rasterio.open(RGB) as r, rasterio.open(MS) as m, rasterio.open(DSM) as dz:
    with WarpedVRT(dz,crs=GRID['crs'],transform=GRID['transform'],width=GRID['width'],
                   height=GRID['height'],resampling=Resampling.bilinear) as zv:
        for row in CIRCLES.itertuples():
            cid=int(row.circle_id);total=0;common=0;mscount=0;rgbcount=0;dsmcount=0
            for win in circle_windows(row.geometry,crop=256,stride=256):
                x,v=read_patch(win,r,m,zv);_,inside=label_patch(win,row.geometry)
                total+=int(inside.sum());common+=int((inside&v).sum())
                # No need for separate estimates: inspect common support first.
            frac=common/max(total,1)
            COVERAGE.append(dict(circle_id=cid,fold=FOLD_BY_ID[cid],n_inside=total,
                                 n_common=common,common_fraction=frac,
                                 n_windows=len(circle_windows(row.geometry,crop=256,stride=192))))
            print('CIRCLE',cid,'fold',FOLD_BY_ID[cid],'common coverage',round(frac,3),
                  'window count',COVERAGE[-1]['n_windows'],flush=True)
coverage=pd.DataFrame(COVERAGE);display(coverage)
coverage.to_csv(OUT/'05_common_coverage_audit.csv',index=False)
print('Minimum circle common coverage',coverage.common_fraction.min())


## 03 — Prepare matched labelled patches; fold-independent cache

The diagnostic gate is intentionally explicit. If the common coverage is poor, do not train: first inspect MS transparency, geometry and DSM alignment.


In [ ]:
MIN_CIRCLE_COVERAGE=0.75
assert (coverage.common_fraction>=MIN_CIRCLE_COVERAGE).all(), (
    'At least one of seven circles has insufficient common RGB/MS/DSM coverage. '
    'Inspect Section 02; do not silently drop circles.')
SEED=20261020;CROP=256;STRIDE=192;MIN_WEIGHT_PIXELS=400
MAX_PATCHES_PER_CIRCLE=100
PATCH_CACHE=OUT/'05_matched_circle_patches_v1.npz'
SOURCE_SIGNATURE=hashlib.sha256(json.dumps({
    'rgb':[str(RGB),RGB.stat().st_size,RGB.stat().st_mtime_ns],
    'ms':[str(MS),MS.stat().st_size,MS.stat().st_mtime_ns],
    'dsm':[str(DSM),DSM.stat().st_size,DSM.stat().st_mtime_ns],
    'circle_path':str(A['exhaustive_annotations']['complete_roi_boundaries']),
    'positive_path':str(A['exhaustive_annotations']['positive_polygons']),
    'coverage':coverage[['circle_id','n_common']].values.tolist(),
    'crop':CROP,'stride':STRIDE,'minpix':MIN_WEIGHT_PIXELS,
    },sort_keys=True).encode()).hexdigest()
META=OUT/'05_patches_meta.json'
if PATCH_CACHE.exists() and META.exists() and json.loads(META.read_text()).get('source_key')==SOURCE_SIGNATURE:
    z=np.load(PATCH_CACHE)
    X,Y,W,CID=(z[k] for k in ['X','Y','W','CID'])
    print('REUSED',len(X),'matched chips')
else:
    chips=[];targets=[];weights=[];ids=[]
    with rasterio.open(RGB) as r,rasterio.open(MS) as m,rasterio.open(DSM) as d:
        with WarpedVRT(d,crs=GRID['crs'],transform=GRID['transform'],
                       width=GRID['width'],height=GRID['height'],
                       resampling=Resampling.bilinear) as zv:
            for circle in CIRCLES.itertuples():
                cid=int(circle.circle_id);local=[]
                for win in circle_windows(circle.geometry,CROP,STRIDE):
                    xx,valid=read_patch(win,r,m,zv)
                    y,inside=label_patch(win,circle.geometry)
                    w=inside&valid
                    if w.sum()<MIN_WEIGHT_PIXELS:continue
                    local.append((xx,y,w))
                if len(local)>MAX_PATCHES_PER_CIRCLE:
                    rng=np.random.default_rng(SEED+cid)
                    keep=np.sort(rng.choice(len(local),MAX_PATCHES_PER_CIRCLE,replace=False))
                    local=[local[j] for j in keep]
                print('Circle',cid,'training/eval patches',len(local),flush=True)
                assert local, f'No matched patches for Baego circle {cid}'
                for xx,y,w in local:
                    chips.append(xx.astype('float32'))
                    targets.append(y.astype('uint8'))
                    weights.append(w.astype('uint8'))
                    ids.append(cid)
    X=np.stack(chips).astype('float32')
    Y=np.stack(targets).astype('uint8')
    W=np.stack(weights).astype('uint8')
    CID=np.array(ids,dtype='int16')
    np.savez_compressed(PATCH_CACHE,X=X,Y=Y,W=W,CID=CID)
    META.write_text(json.dumps({'source_key':SOURCE_SIGNATURE,'n_patches':len(X)},indent=2))
print('COMMON PATCHES:',X.shape,'| positive labelled fraction',
      float((Y*W).sum()/max(W.sum(),1)))
print(pd.DataFrame({'circle':CID}).value_counts().sort_index())


## 04 — Comparable models and train-only normalization

Use a ResNet18 U-Net for all configurations. The extra channels use SMP's pretrained-weight adaptation, so RGB-only remains the comparable architecture; training random seeds and examples remain matched. The multispectral/DSM scalers are fit **only on training circles** in each fold.


In [ ]:
import segmentation_models_pytorch as smp
VARIANTS={
    'RGB':[0,1,2],
    'RGB_DSM':[0,1,2,7],
    'RGB_MS':[0,1,2,3,4,5,6],
    'RGB_MS_DSM':[0,1,2,3,4,5,6,7]
}
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
assert DEVICE=='cuda','Select a GPU runtime in Colab for training.'
EPOCHS=15;BATCH=8;LR=2e-4
print('GPU',torch.cuda.get_device_name(0),'models',list(VARIANTS))

def train_scaler(train_ids):
    assert len(train_ids)>0
    rng=np.random.default_rng(SEED)
    chosen=rng.choice(train_ids,size=min(len(train_ids),100),replace=False)
    parts=[]
    for i in chosen:
        valid=W[i].astype(bool)
        px=X[i,:,valid] if False else X[i].reshape(8,-1)[:,valid.ravel()]
        if px.shape[1]>4000: px=px[:,rng.choice(px.shape[1],4000,replace=False)]
        parts.append(px)
    allv=np.concatenate(parts,axis=1)
    lo=np.percentile(allv,2,axis=1).astype('float32')
    hi=np.percentile(allv,98,axis=1).astype('float32')
    lo[:3]=0;hi[:3]=1
    assert np.all(np.isfinite(lo)) and np.all(np.isfinite(hi))
    return lo,hi

class Patches(Dataset):
    def __init__(self,ids,channels,lo,hi,augment):
        self.ids=ids;self.ch=channels;self.lo=lo;self.hi=hi;self.augment=augment
    def __len__(self):return len(self.ids)
    def __getitem__(self,k):
        j=int(self.ids[k])
        x=X[j,self.ch].copy()
        lo=self.lo[self.ch,None,None];hi=self.hi[self.ch,None,None]
        x=np.clip((x-lo)/np.maximum(hi-lo,1e-6),0,1)
        y=Y[j].astype('float32');w=W[j].astype('float32')
        if self.augment:
            kk=np.random.randint(0,4)
            x=np.rot90(x,kk,axes=(1,2))
            y=np.rot90(y,kk);w=np.rot90(w,kk)
            if np.random.rand()<0.5:
                x=x[:,:,::-1];y=y[:,::-1];w=w[:,::-1]
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.from_numpy(np.ascontiguousarray(y)),
                torch.from_numpy(np.ascontiguousarray(w)))


## 05 — Three-fold Baego-only ablation (12 training runs)

**Set RUN_TRAIN=True** only after reviewing common-coverage audit and valid multispectral pixels. Reports per-epoch progress and checkpoints so reruns skip completed folds. Circle-level evaluation uses the same held-out patch positions in every variant. Pixels are counted once per patch (overlapping pixels may occur more than once); this is an exploratory patch-based pilot, not a fully de-duplicated geographic mosaic evaluation.


In [ ]:
RUN_TRAIN=False
def model_for(channels):
    return smp.Unet(encoder_name='resnet18',encoder_weights='imagenet',
                    in_channels=len(channels),classes=1).to(DEVICE)

def masked_bce(logit,y,w):
    b=F.binary_cross_entropy_with_logits(logit[:,0].float(),y,reduction='none')
    return (b*w).sum()/w.sum().clamp_min(1.)

def evaluate_one(model,valid_ids,variant,fold,lo,hi):
    model.eval();acc={}
    ds=Patches(valid_ids,VARIANTS[variant],lo,hi,augment=False)
    loader=DataLoader(ds,batch_size=BATCH,shuffle=False,num_workers=0)
    cursor=0
    with torch.inference_mode():
        for x,y,w in loader:
            p=torch.sigmoid(model(x.to(DEVICE))[:,0].float()).cpu().numpy()
            yy=y.numpy();ww=w.numpy().astype(bool)
            for j in range(len(p)):
                cid=int(CID[valid_ids[cursor]])
                if cid not in acc:acc[cid]=dict(n=0,truth=0.,pred=0.,tp=0,fp=0,fn=0,
                                              brier=0.)
                a=acc[cid];mask=ww[j]
                t=yy[j][mask];q=p[j][mask];binary=q>=0.5
                a['n']+=len(t);a['truth']+=float(t.sum());a['pred']+=float(q.sum())
                a['tp']+=int(((t==1)&binary).sum())
                a['fp']+=int(((t==0)&binary).sum())
                a['fn']+=int(((t==1)&~binary).sum())
                a['brier']+=float(np.square(q-t).sum())
                cursor+=1
    rows=[]
    for cid,a in acc.items():
        n=a['n'];obs=100*a['truth']/n;pred=100*a['pred']/n
        rows.append(dict(variant=variant,fold=fold,circle_id=cid,
             n_pixels_patch_weighted=n,observed_fraction=obs,predicted_fraction=pred,
             abs_error_pp=abs(pred-obs),bias_pp=pred-obs,
             pixel_f1=2*a['tp']/max(1,2*a['tp']+a['fp']+a['fn']),
             pixel_iou=a['tp']/max(1,a['tp']+a['fp']+a['fn']),
             brier=a['brier']/n))
    return rows

if RUN_TRAIN:
    ALL=[]
    for fold in [0,1,2]:
        train_ids=np.where(np.array([FOLD_BY_ID[int(v)]!=fold for v in CID]))[0]
        valid_ids=np.where(np.array([FOLD_BY_ID[int(v)]==fold for v in CID]))[0]
        assert len(set(CID[train_ids])&set(CID[valid_ids]))==0
        lo,hi=train_scaler(train_ids)
        print('\n=== FOLD',fold,'train circles',sorted(set(CID[train_ids])),
              'holdout',sorted(set(CID[valid_ids])),'===',flush=True)
        for variant,ch in VARIANTS.items():
            ckpt=OUT/f'{variant}_fold{fold}_v1.pt'
            torch.manual_seed(SEED+fold)
            np.random.seed(SEED+fold);random.seed(SEED+fold)
            model=model_for(ch)
            if ckpt.exists():
                state=torch.load(ckpt,map_location=DEVICE,weights_only=True)
                model.load_state_dict(state)
                print('Reuse checkpoint:',ckpt.name,flush=True)
            else:
                ds=Patches(train_ids,ch,lo,hi,augment=True)
                loader=DataLoader(ds,batch_size=BATCH,shuffle=True,num_workers=0)
                optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
                scaler=torch.amp.GradScaler('cuda')
                for epoch in range(EPOCHS):
                    tic=time.time();model.train();losses=[]
                    for x,y,w in loader:
                        x=x.to(DEVICE);y=y.to(DEVICE);w=w.to(DEVICE)
                        optimizer.zero_grad(set_to_none=True)
                        with torch.autocast('cuda',dtype=torch.float16):
                            logits=model(x)
                        loss=masked_bce(logits,y,w)
                        scaler.scale(loss).backward()
                        scaler.step(optimizer);scaler.update()
                        losses.append(float(loss.item()))
                    print(f'{variant} fold={fold} epoch={epoch+1:02}/{EPOCHS} '
                          f'BCE={np.mean(losses):.4f} elapsed={time.time()-tic:.1f}s',
                          flush=True)
                torch.save(model.state_dict(),ckpt)
            ALL.extend(evaluate_one(model,valid_ids,variant,fold,lo,hi))
            pd.DataFrame(ALL).to_csv(OUT/'05_pilot_circle_progress.csv',index=False)
            del model;torch.cuda.empty_cache();gc.collect()
    scores=pd.DataFrame(ALL)
    scores.to_csv(OUT/'05_baego_multimodal_3fold_circle_scores.csv',index=False)
    print('Completed',len(scores),'held-out circle × model results')
else:
    print('AUDIT COMPLETE; set RUN_TRAIN=True after checking coverage.')


## 06 — Paired model summary, per-circle scatterplots and gains over RGB


In [ ]:
SCORES_FILE=OUT/'05_baego_multimodal_3fold_circle_scores.csv'
if not SCORES_FILE.exists():
    print('No finished CV scores yet — run Section 05 first.')
else:
    s=pd.read_csv(SCORES_FILE)
    assert s.groupby('variant').circle_id.nunique().to_dict()=={
        name:7 for name in VARIANTS}
    summary=s.groupby('variant').agg(circles=('circle_id','nunique'),
        MAE_pp=('abs_error_pp','mean'),bias_pp=('bias_pp','mean'),
        mean_pixel_F1=('pixel_f1','mean'),mean_pixel_IoU=('pixel_iou','mean'),
        Brier=('brier','mean')).sort_values('MAE_pp')
    base=float(summary.loc['RGB','MAE_pp'])
    summary['MAE_improvement_over_RGB_pp']=base-summary.MAE_pp
    display(summary.round(4))
    summary.to_csv(OUT/'05_baego_multimodal_summary.csv')
    fig,axes=plt.subplots(1,4,figsize=(16,4),sharex=True,sharey=True)
    for ax,(variant,g) in zip(axes,s.groupby('variant',sort=False)):
        ax.scatter(g.observed_fraction,g.predicted_fraction,s=50)
        ax.plot([0,100],[0,100],'k--',lw=1)
        ax.set(xlim=(0,100),ylim=(0,100),title=variant,
               xlabel='Annotated Musanga cover (%)')
        ax.grid(alpha=.2)
    axes[0].set_ylabel('Predicted Musanga cover (%)')
    fig.suptitle('Baego circles: held-out 3-fold spatial CV (matched sensor coverage)')
    fig.tight_layout()
    fig.savefig(OUT/'05_baego_multimodal_scatter.png',dpi=160)
    plt.show()
    wide=s.pivot(index='circle_id',columns='variant',values='abs_error_pp')
    display(wide.round(2))
    print('NOTE: seven circles only; report paired per-circle differences,'
          ' not cell-level significance.')
